# **XGBoost**

это реализация градиентного бустинга над деревьями решений с регуляризацией, которая последовательно добавляет новые деревья, уменьшая ошибку предыдущих и тем самым строя сильную модель из множества слабых.


In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score, classification_report
import xgboost as xgb


Таргет Good trade (То есть ret_H — вспомогательная переменная для вычисления GoodTrade)


In [ ]:
def add_goodtrade_target(df, horizon=20):
    df = df.copy()

    # Будущая цена
    df['Close_fwd'] = df['Close'].shift(-horizon)

    # Доходность по направлению сигнала
    ret_long = (df['Close_fwd'] - df['Close']) / df['Close']
    ret_short = (df['Close'] - df['Close_fwd']) / df['Close']

    ret = np.where(
        df['EntrySignal'] > 0, ret_long,
        np.where(df['EntrySignal'] < 0, ret_short, 0.0)
    )

    df['ret_H'] = ret

    # GoodTrade: есть сигнал и через H баров прибыль > 0
    df['GoodTrade'] = ((df['EntrySignal'] != 0) & (df['ret_H'] > 0)).astype(int)

    # убираем последние horizon строк (там нет Close_fwd)
    df = df.iloc[:-horizon]

    return df


In [ ]:
df = pd.read_csv("Brent.csv")
df = add_goodtrade_target(df, horizon=20)


In [ ]:
cols_nan = ["AddOn_Anchor_Level", "AddOn_Anchor_IsUp", "AddOn_Size_Pct"]

df = df.dropna(subset=cols_nan).reset_index(drop=True)


In [ ]:
df

,DateTime,Open,High,Low,Close,Alligator_Jaw,Alligator_Teeth,Alligator_Lips,Fractal_Up,Fractal_Down,...,Fractal_Up_conf,Fractal_Down_conf,AddOn_Anchor_Level,AddOn_Anchor_IsUp,AddOn_Size_Pct,AddOn_Ready,AddOn_Triggered,Close_fwd,ret_H,GoodTrade
0,2015-10-26 19:00:00,47.76,48.00,47.70,47.86,48.15240,48.12767,48.01926,1,0,...,0,1,47.70,0.0,0.3,1,1,47.08,0.00000,0
1,2015-10-26 20:00:00,47.86,47.91,47.54,47.65,48.16106,48.10359,47.94841,0,0,...,0,0,47.70,0.0,0.3,0,0,47.10,0.00000,0
2,2015-10-26 21:00:00,47.65,47.82,47.52,47.57,48.16367,48.05876,47.91373,0,0,...,1,0,47.70,0.0,0.3,0,0,47.16,0.00000,0
3,2015-10-26 22:00:00,47.57,47.59,47.43,47.47,48.16070,48.00954,47.90098,0,0,...,0,0,47.70,0.0,0.3,0,0,47.17,0.00632,1
4,2015-10-26 23:00:00,47.47,47.51,47.31,47.31,48.14334,47.98022,47.86578,0,0,...,0,0,47.70,0.0,0.3,0,0,47.10,0.00000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36210,2025-10-20 18:00:00,60.73,61.04,60.51,60.60,61.31318,61.20603,61.06995,0,0,...,0,1,63.53,0.0,0.3,0,0,61.65,0.00000,0
36211,2025-10-20 19:00:00,60.68,60.79,60.63,60.73,61.29409,61.16590,60.96996,0,0,...,0,0,63.53,0.0,0.3,0,0,62.46,0.00000,0
36212,2025-10-20 20:00:00,60.73,61.06,60.66,60.98,61.28646,61.13203,60.92697,0,0,...,0,0,63.53,0.0,0.3,0,0,62.35,0.00000,0
36213,2025-10-20 21:00:00,60.98,61.14,60.92,60.96,61.27366,61.06178,60.89658,1,0,...,0,0,63.53,0.0,0.3,0,0,62.36,0.00000,0


Кодируем признаки

In [ ]:
# 1. Заполняем пропуски и делаем категорию
df['EntryReason'] = df['EntryReason'].fillna('None').astype('category')

# 2. Смотрим маппинг кодов (для контроля, можно один раз глянуть)
entryreason_mapping = dict(enumerate(df['EntryReason'].cat.categories))
print("Mapping EntryReason:", entryreason_mapping)

# 3. Перезаписываем EntryReason числовыми кодами
df['EntryReason'] = df['EntryReason'].cat.codes.astype('int16')

# 4. Удаляем лишние текстовые столбцы, если они есть
df = df.drop(columns=['EntryReason_raw', 'EntryReason_code'], errors='ignore')

# 5. Проверяем, что не осталось других строк/категорий
print("Нечисловые колонки:",
      df.select_dtypes(include=['object', 'category']).columns.tolist())


Mapping EntryReason: {0: 'None', 1: 'saucer', 2: 'three_color', 3: 'zero_cross'}
Нечисловые колонки: ['DateTime']


Подготовка признаков

In [ ]:
import numpy as np

def prepare_train_data(df, target_col='GoodTrade', signals_only=True):
    df = df.copy()

    # берём только бары, где есть сигнал (мы фильтруем входы)
    if signals_only:
        df = df[df['EntrySignal'] != 0]

    # колонки, которые точно НЕ идут в признаки
    drop_cols = ['DateTime', 'Close_fwd', 'ret_H']
    drop_cols = [c for c in drop_cols if c in df.columns]

    # кандидаты в признаки
    candidate_cols = [
        c for c in df.columns
        if c not in drop_cols + [target_col]
    ]

    # оставляем только числовые признаки
    feature_cols = df[candidate_cols].select_dtypes(include=[np.number]).columns.tolist()

    X = df[feature_cols].astype('float32').values
    y = df[target_col].values

    return X, y, feature_cols, df


Подготовка дынных

In [ ]:
X, y, feature_cols, df_signals = prepare_train_data(df)

split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print("X_train dtype:", X_train.dtype)
print("X_train shape:", X_train.shape)


X_train dtype: float32
X_train shape: (1430, 31)


настройка баланса классов

In [ ]:
# баланс классов
pos = (y_train == 1).sum()
neg = (y_train == 0).sum()
scale_pos_weight = neg / pos if pos > 0 else 1.0
print(f"pos={pos}, neg={neg}, scale_pos_weight={scale_pos_weight:.2f}")


pos=701, neg=729, scale_pos_weight=1.04


Модель

In [ ]:
model = xgb.XGBClassifier(
    objective='binary:logistic',
    n_estimators=1000,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_lambda=2.0,
    scale_pos_weight=scale_pos_weight,
    tree_method="hist",
    eval_metric="auc",
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],   # будем смотреть AUC на тесте
    verbose=50                     # печатать лог каждые 50 итераций
)



[0]	validation_0-auc:0.51445
[50]	validation_0-auc:0.54657
[100]	validation_0-auc:0.55216
[150]	validation_0-auc:0.55943
[200]	validation_0-auc:0.54849
[250]	validation_0-auc:0.53724
[300]	validation_0-auc:0.53451
[350]	validation_0-auc:0.53663
[400]	validation_0-auc:0.53510
[450]	validation_0-auc:0.53723
[500]	validation_0-auc:0.53395
[550]	validation_0-auc:0.53179
[600]	validation_0-auc:0.53157
[650]	validation_0-auc:0.53182
[700]	validation_0-auc:0.52951
[750]	validation_0-auc:0.52954
[800]	validation_0-auc:0.52389
[850]	validation_0-auc:0.52232
[900]	validation_0-auc:0.51883
[950]	validation_0-auc:0.52236
[999]	validation_0-auc:0.52111


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.7, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='auc', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=3,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=1000,
              n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

# вероятности того, что сделка хорошая
proba_test = model.predict_proba(X_test)[:, 1]

# можно взять порог 0.5 (потом поиграемся)
y_pred = (proba_test >= 0.5).astype(int)

print("AUC на тесте:", roc_auc_score(y_test, proba_test))
print()
print("Отчёт по классификации:")
print(classification_report(y_test, y_pred))

print("Матрица ошибок:")
print(confusion_matrix(y_test, y_pred))


AUC на тесте: 0.5211060771754413

Отчёт по классификации:
              precision    recall  f1-score   support

           0       0.53      0.60      0.56       185
           1       0.51      0.44      0.47       173

    accuracy                           0.52       358
   macro avg       0.52      0.52      0.52       358
weighted avg       0.52      0.52      0.52       358

Матрица ошибок:
[[111  74]
 [ 97  76]]


In [ ]:
import pandas as pd
import numpy as np

importances = model.feature_importances_
fi = pd.DataFrame({'feature': feature_cols, 'importance': importances})
fi = fi.sort_values('importance', ascending=False)
fi.head(20)
